In [11]:
from box_sdk_gen import BoxClient, BoxDeveloperTokenAuth
from dotenv import load_dotenv
import os
import logging

logger = logging.getLogger()
logger.setLevel(logging.INFO)

load_dotenv("box_access_dev.env")  # loads variables from box_access_dev.env into environment

token = os.getenv("BOX_TOKEN")
def login(token: str):
    auth: BoxDeveloperTokenAuth = BoxDeveloperTokenAuth(token=token)
    client: BoxClient = BoxClient(auth=auth)
    return client

client = login(token)

In [12]:
def get_folder_by_path(client, path):
    """
    path example: "Klein Lab/Fabrication/Exfoliation/"
    """
    current_folder_id = "0"  # Root

    for folder_name in path.strip("/").split("/"):
        found = False

        items = client.folders.get_folder_items(current_folder_id).entries

        for item in items:
            if item.type == "folder" and item.name == folder_name:
                current_folder_id = item.id
                found = True
                break

        if not found:
            raise Exception(f"Folder '{folder_name}' not found")

    return current_folder_id

In [13]:
def download_box_file(file_id: str, local_file_path: str):
    """
    Downloads a file from Box using box-sdk-gen.

    :param file_id: The ID of the file to download.
    :param local_file_path: The local path to save the file to.
    :param token: Your Box developer token.
    """
    try:
        # Download the file content as a stream
        file_content_stream = client.downloads.download_file(file_id)

        # Write the stream to a local file in binary mode
        with open(local_file_path, 'wb') as local_file:
            for chunk in file_content_stream:
                local_file.write(chunk)
        
        logging.info(f"File '{file_id}' successfully downloaded to '{local_file_path}'")

    except Exception as e:
        logging.error(f"An error occurred: {e}")

In [14]:
exfoliation_path = "Klein Lab/Fabrication/Exfoliation/"
users_folder_id = get_folder_by_path(client, "Klein Lab/Fabrication/Exfoliation/")
users = client.folders.get_folder_items(users_folder_id).entries

for item in users:
    if item.type == "folder": # enter into folder of user
        user_images = client.folders.get_folder_items(item.id).entries
        for possible_image in user_images:
            if possible_image.type == "file" and ".jpg" in possible_image.name:
                download_box_file(possible_image.id, "download.jpg")
                break


INFO:root:File '2135174720700' successfully downloaded to 'download.jpg'
INFO:root:File '2134284406957' successfully downloaded to 'download.jpg'
